In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder\
        .appName("scenario_4")\
        .master("local[*]")\
        .config("spark.driver.memory","6g")\
        .config("spark.executtor.memory","6g")\
        .config("spark.sql.shuffle.partitions","50")\
        .config("saprk.sql.adaptive.enabled","true")\
        .config("spark.sql.autoBroadcastJoinThreshold","-1")\
        .config("spakr.python.worker.reuse","false")\
        .config("spark.executor.heartbeatInterval","60s")\
        .config("spark.network.timeout","300s")\
        .config("spark.hadoop.io.native.lib.available","false")\
        .config("spark.serializer","org.apache.spark.serializer.KryoSerializer")\
        .getOrCreate()


In [2]:
v1 = "E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/v1/"
v2 = "E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/v2/"

events_dir = "E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/*/"

In [3]:
df1=spark.read.parquet(events_dir)
print("=== Schema ====")
df1.printSchema()
df1.select(
    when(col('ab_test_group').isNull(),1).alias("null_ab_values"),
    when(col('feature_flag').isNull(),1).alias("null_flag_values")
).show()

=== Schema ====
root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp_ntz (nullable = true)
 |-- page_name: string (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- device_type: string (nullable = true)
 |-- os: string (nullable = true)
 |-- country: string (nullable = true)
 |-- app_version: string (nullable = true)



AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `ab_test_group` cannot be resolved. Did you mean one of the following? [`event_id`, `event_type`, `app_version`, `country`, `page_name`].;
'Project [CASE WHEN isnull('ab_test_group) THEN 1 END AS null_ab_values#22, CASE WHEN isnull('feature_flag) THEN 1 END AS null_flag_values#23]
+- Relation [event_id#0,user_id#1,session_id#2,event_type#3,event_timestamp#4,page_name#5,duration_seconds#6,device_type#7,os#8,country#9,app_version#10] parquet


In [4]:
df1=spark.read.option("mergeSchema",True).parquet(events_dir)
print("=== Schema ====")
df1.printSchema()
df_1=df1.select(
    count('*').alias("total_rows"),
    count(when(col('ab_test_group').isNull(),1)).alias("null_ab_values"),
    count(when(col('feature_flag').isNull(),1)).alias("null_flag_values")
)
df_1.show()
df1.groupBy('ab_test_group').agg(count('*').alias("total_counts")).show()

=== Schema ====
root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp_ntz (nullable = true)
 |-- page_name: string (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- device_type: string (nullable = true)
 |-- os: string (nullable = true)
 |-- country: string (nullable = true)
 |-- app_version: string (nullable = true)
 |-- ab_test_group: string (nullable = true)
 |-- feature_flag: string (nullable = true)

+----------+--------------+----------------+
|total_rows|null_ab_values|null_flag_values|
+----------+--------------+----------------+
|  10000000|       5000000|         5000000|
+----------+--------------+----------------+

+-------------+------------+
|ab_test_group|total_counts|
+-------------+------------+
|         NULL|     5000000|
|    variant_a|     1667050|
|      control|     1665237|
|    variant_b|     1667